# Practical 4: Label Requests as Benign or Attack
**Course:** BE05000231 - PDS (Semester V, GTU GSET)
**Course Outcome:** CO2

### Objective:
Label HTTP requests as benign or specific attack classes (SQLi, XSS, Path Traversal, Scanners, Brute Force) using YAML signature rules mapped to OWASP Top 10 and MITRE ATT&CK, coupled with behavioral sliding-window detection and ground-truth verification.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')
import pandas as pd

from pipeline.parser import parse_file
from pipeline.cleaner import clean
from pipeline.labeler import load_rules, label_requests, ground_truth_label, evaluate_labeling

LOG_PATH = Path('../tests/fixtures/sample_access.log')
RULES_PATH = Path('../rules/web_rules.yaml')


## 2. Ingest, Parse, and Clean Access Logs

In [ ]:
raw_df, _ = parse_file(LOG_PATH)
cleaned_df = clean(raw_df)
print(f'Total cleaned records: {len(cleaned_df)}')


## 3. Load Detection Rules & Map to Threat Frameworks

In [ ]:
rules = load_rules(RULES_PATH)
print(f'Loaded {len(rules)} signature rules:')
for r in rules:
    print(f"  [{r['id']}] {r['name']} -> Label: {r['label']}, Severity: {r['severity']}, OWASP: {r.get('owasp')}, MITRE: {r.get('mitre')}")


## 4. Run Rule Matching & Sliding-Window Brute Force Detection

In [ ]:
labeled_df = label_requests(cleaned_df, rules, bf_threshold=5, bf_window='1min')
print('Class Distribution:')
print(labeled_df['label'].value_counts())


## 5. False Positive Trap Verification
A critical check in cyber-security log analysis: ensure harmless search queries containing keywords like `union station` do NOT get falsely labeled as SQL injection.

In [ ]:
trap_row = labeled_df[labeled_df['raw_url'].str.contains('union\+station')]
print('Search query trap row:')
print(trap_row[['ip', 'raw_url', 'label', 'rule_ids']].to_dict(orient='records'))


## 6. Explainability: Inspect Rule Firing Traces
Every detected alert records the exact rule IDs that triggered it in `rule_ids`.

In [ ]:
sample_attacks = labeled_df[labeled_df['label'] != 'benign'][['ip', 'method', 'raw_url', 'label', 'rule_ids']].head(10)
sample_attacks


## 7. Ground Truth Fusion and Evaluation

In [ ]:
# Simulated ground truth from controlled lab capture
gt_records = pd.DataFrame([
    {'attacker_ip': '198.51.100.7', 'start_ts': '2026-10-10T13:56:00Z', 'end_ts': '2026-10-10T13:56:20Z', 'attack_type': 'sqli'},
    {'attacker_ip': '198.51.100.12', 'start_ts': '2026-10-10T13:56:20Z', 'end_ts': '2026-10-10T13:56:32Z', 'attack_type': 'path_traversal'},
    {'attacker_ip': '192.0.2.44', 'start_ts': '2026-10-10T13:57:00Z', 'end_ts': '2026-10-10T13:57:30Z', 'attack_type': 'brute_force'},
])
fused_df = ground_truth_label(labeled_df, gt_records)
metrics = evaluate_labeling(fused_df, pred_col='label', true_col='ground_truth')
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1_score']:.4f}")
print(f"Confusion Matrix:\n{pd.DataFrame(metrics['confusion_matrix'])}")


### Conclusion:
Practical 4 demonstrated explainable signature-based and behavioral labeling mapped to industry standard OWASP Top 10 and MITRE ATT&CK taxonomies, avoiding circular reasoning by validating against independent ground-truth data.